In [46]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader
from typing import List
from langchain_classic.schema import Document

In [37]:
file_path = Path(r"C:\Users\amanm\Desktop\learning\rag-project-2\data\2506.08276.pdf")

## loading the document

In [38]:
def document_loader(file_path : Path):
    file_format = file_path.suffix.lower()
    if file_format == ".pdf":
        docs = PyMuPDFLoader(file_path=str(file_path))
    elif file_format == ".txt":
        docs = TextLoader(file_path=str(file_path), encoding="utf-8")
    else:
        raise ValueError(f"Unsupported document format: {file_format}")
    return docs.load()


In [43]:
d = document_loader(file_path=file_path)

## Parent Chunk

In [47]:
def parent_splitter(documents : List[Document], chunk_size : int, chunk_overlap : int):
        splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
        parent_chunks = splitter.split_documents(documents)
        return parent_chunks

    

In [62]:
parent_chunks = parent_splitter(d, 2000, 200)

## children splitter


In [63]:
def children_splitter(parent_chunks : List[Document], chunk_size : int, chunk_overlap : int):
    splitter = RecursiveCharacterTextSplitter(chunk_size = chunk_size, chunk_overlap = chunk_overlap)
    child_chunks = splitter.split_documents(parent_chunks)
    return child_chunks

In [65]:
children_splitter(parent_chunks, 200, 20)

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:4177c2c)', 'creationdate': '', 'source': 'C:\\Users\\amanm\\Desktop\\learning\\rag-project-2\\data\\2506.08276.pdf', 'file_path': 'C:\\Users\\amanm\\Desktop\\learning\\rag-project-2\\data\\2506.08276.pdf', 'total_pages': 17, 'format': 'PDF 1.7', 'title': 'LEANN: A Low-Storage Vector Index', 'author': 'Yichuan Wang; Zhifei Li; Shu Liu; Yongji Wu; Ziming Mao; Yilong Zhao; Xiao Yan; Zhiying Xu; Yang Zhou; Ion Stoica; Sewon Min; Matei Zaharia; Joseph E. Gonzalez', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='LEANN: A LOW-STORAGE OVERHEAD VECTOR INDEX\nYichuan Wang† 1 Zhifei Li 1 Shu Liu 1 Yongji Wu† 1 Ziming Mao 1 Yilong Zhao 1 Xiao Yan 2 Zhiying Xu∗3'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:4177c2c)', 'creationdate': '', 'source': 'C:\\Users\\amanm\\Desktop\\learning\\rag-project-2\\da

## attaching parent - id metadata

In [ ]:
def create_parent_child_mapping(parent_chunks : List[Document]):

    for idx,chunk in enumerate(parent_chunks):
        chunk.metadata["parent_id"] = f"Parent-{idx+1}"
    return parent_chunks,children_splitter(parent_chunks, 200, 20)

In [73]:
parent_chunks, child_chunks = create_parent_child_mapping(parent_splitter(d, 1000, 100))

In [100]:
child_chunks[0]

Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:4177c2c)', 'creationdate': '', 'source': 'C:\\Users\\amanm\\Desktop\\learning\\rag-project-2\\data\\2506.08276.pdf', 'file_path': 'C:\\Users\\amanm\\Desktop\\learning\\rag-project-2\\data\\2506.08276.pdf', 'total_pages': 17, 'format': 'PDF 1.7', 'title': 'LEANN: A Low-Storage Vector Index', 'author': 'Yichuan Wang; Zhifei Li; Shu Liu; Yongji Wu; Ziming Mao; Yilong Zhao; Xiao Yan; Zhiying Xu; Yang Zhou; Ion Stoica; Sewon Min; Matei Zaharia; Joseph E. Gonzalez', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'parent_id': 'Parent-1'}, page_content='LEANN: A LOW-STORAGE OVERHEAD VECTOR INDEX\nYichuan Wang† 1 Zhifei Li 1 Shu Liu 1 Yongji Wu† 1 Ziming Mao 1 Yilong Zhao 1 Xiao Yan 2 Zhiying Xu∗3')

## Parent-Child RAG needs two storage systems. Indoc store for parents and embeddings for children

In [135]:
# parent store (dictionary)

parent_store = {}

for idx,pchunks in enumerate(parent_chunks):
    parent_store[f"Parent-{idx+1}"] = pchunks.page_content



In [115]:
# metadata, child_id, parent_id, source, page
from langchain_core.documents import Document
doc = []
for idx,cchunks in enumerate(child_chunks):
    d = {}
    d["child_id"] = idx + 1
    d['source'] =  Path(cchunks.metadata['source']).name
    d['page'] = cchunks.metadata['page']
    d['parent_id'] = cchunks.metadata['parent_id']

    doc.append(Document(
    page_content=cchunks.page_content,
    metadata=d)
)
print(doc)


[Document(metadata={'child_id': 1, 'source': '2506.08276.pdf', 'page': 0, 'parent_id': 'Parent-1'}, page_content='LEANN: A LOW-STORAGE OVERHEAD VECTOR INDEX\nYichuan Wang† 1 Zhifei Li 1 Shu Liu 1 Yongji Wu† 1 Ziming Mao 1 Yilong Zhao 1 Xiao Yan 2 Zhiying Xu∗3'), Document(metadata={'child_id': 2, 'source': '2506.08276.pdf', 'page': 0, 'parent_id': 'Parent-1'}, page_content='Yang Zhou 1 4 Ion Stoica 1 Sewon Min 1 Matei Zaharia 1 Joseph E. Gonzalez 1\nABSTRACT\nEmbedding-based vector search underpins many important applications, such as recommendation and retrieval-'), Document(metadata={'child_id': 3, 'source': '2506.08276.pdf', 'page': 0, 'parent_id': 'Parent-1'}, page_content='augmented generation (RAG). It relies on vector indices to enable efficient search. However, these indices require'), Document(metadata={'child_id': 4, 'source': '2506.08276.pdf', 'page': 0, 'parent_id': 'Parent-1'}, page_content='storing high-dimensional embeddings and large index metadata, whose total size can 

In [ ]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os
load_dotenv()


pc = Pinecone(api_key="")

index_name = "integrated-dense-py"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

In [117]:
index = pc.Index(index_name)

In [111]:
doc[0]

Document(metadata={'child_id': 1, 'source': 'C:\\Users\\amanm\\Desktop\\learning\\rag-project-2\\data\\2506.08276.pdf', 'page': 0, 'parent_id': 'Parent-1'}, page_content='LEANN: A LOW-STORAGE OVERHEAD VECTOR INDEX\nYichuan Wang† 1 Zhifei Li 1 Shu Liu 1 Yongji Wu† 1 Ziming Mao 1 Yilong Zhao 1 Xiao Yan 2 Zhiying Xu∗3')

In [118]:
records = []

for d in doc:
    records.append({
        "_id": f"child-{d.metadata['child_id']}",
        "chunk_text": d.page_content,
            "parent_id": d.metadata["parent_id"],
            "source": d.metadata["source"],
            "page": d.metadata["page"]
    })

In [119]:
records[0]

{'_id': 'child-1',
 'chunk_text': 'LEANN: A LOW-STORAGE OVERHEAD VECTOR INDEX\nYichuan Wang† 1 Zhifei Li 1 Shu Liu 1 Yongji Wu† 1 Ziming Mao 1 Yilong Zhao 1 Xiao Yan 2 Zhiying Xu∗3',
 'parent_id': 'Parent-1',
 'source': '2506.08276.pdf',
 'page': 0}

In [120]:
for i in range(0, len(records), 96):
        batch = records[i:i+96]
        index.upsert_records(namespace="default", records=batch)


# query



In [151]:
results = index.search(
    namespace="default",
    query={
        "inputs": {"text": "LEAN is"},
        "top_k": 5
    }
)

parent_ids = set()
context_chunks = []
sources = []

for result in results.result.hits:

    pid = result.fields["parent_id"]

    if pid not in parent_ids:
        parent_ids.add(pid)
        context_chunks.append(parent_store[pid])
    
    sources.append({
            "source": result.fields["source"],
            "page": result.fields["page"]
        })

context = "\n\n".join(context_chunks)

print(context[:500])
print(sources)

have been visited. Empirically, graph-based indexes achieve
high recall with only O(log N) embedding extractions and
distance computations. This is because the graph traver-
sal can quickly converge on the neighbors of the query by
moving to more similar neighbors in each step.
3
LEANN OVERVIEW
Figure 1 shows the end-to-end workflow of LEANN, which
includes offline index construction and online query serving.
Offline stage Given a dataset of items, such as chunked un-
structured text, LEANN comp
[{'source': '2506.08276.pdf', 'page': 2.0}, {'source': '2506.08276.pdf', 'page': 2.0}, {'source': '2506.08276.pdf', 'page': 1.0}, {'source': '2506.08276.pdf', 'page': 15.0}, {'source': '2506.08276.pdf', 'page': 3.0}]


In [152]:
from src.chunking.parent_child import ingest

In [154]:
from pathlib import Path

In [155]:
file_path = Path(r"C:\Users\amanm\Desktop\learning\rag-project-2\data\2506.08276.pdf")

In [157]:
records, parents = ingest(file_path=file_path)

In [159]:
records[0]

{'_id': 'child-1',
 'chunk_text': 'LEANN: A LOW-STORAGE OVERHEAD VECTOR INDEX\nYichuan Wang† 1 Zhifei Li 1 Shu Liu 1 Yongji Wu† 1 Ziming Mao 1 Yilong Zhao 1 Xiao Yan 2 Zhiying Xu∗3',
 'parent_id': 'Parent-1',
 'source': '2506.08276.pdf',
 'page': 0,
 'source_hash_value': '21dc26303357c6381377f0190948a599b36402ded9248e654ef59347c9459ae9'}